# TRAITEMENT DES DONNES FUSIONNEES DE LA PREMIERE FEUILLE EXCEL

**Objectif** : Traitement/apurement des variables démographiques
* Date de naissance 
* Sexe
* Situation Matrimoniale
* Nombre d'enfants

Nous aurons également pour mission de créer de nouvelles variables à partir des informations disponibles. Il s'agit du statut dans l'emploi (Fonctionnaire ou contractuelle), l'âge, l'age de première prise de service, l'ancienneté dans l'organisme, nombre d'année d'expérience.

Un contrôle sera effectué sur le rattachement administratif:

* Matricule
* Fonction
* Services
* Organismes
* Situation dans l'emploi
* Emplois
* Fonction
* Classes/Echelons
* Lieu d'affectation

## Chargement des packages necessaires

In [1]:
# Paramètres
import io
import pandas as pd
import boto3
import io
import unicodedata
import re
import numpy as np
import unicodedata
import unicodedata, re
import re

## Chargement de la base de travail  

In [2]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
 aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "admindataanstat"

# object_key 
object_key = "Solde/panel_solde_df_1_code.parquet"  # Chemin dans le bucket

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
panel_solde_df = pd.read_parquet(io.BytesIO(obj['Body'].read()))

# Afficher un aperçu
panel_solde_df.head()

,MATRICULE||CODE_ORGANISME,MATRICULE,NOM,DATE NAISSANCE,SEXE,SITUATION MATRIMONIALE,NOMBRE ENFANT,LIEU AFFECTATION,SERVICE,ORGANISME,...,SERVICE_NORM,CODE_SERVICE,ORGANISME_NORM,CODE_ORGANISME,CLASSE/ECHELON_NORM,CODE_CLASSE/ECHELON,EMPLOI_NORM,CODE_EMPLOI,SITUATION_NORM,CODE_SITUATION
0,011242X15,011242X,DOSSO MEGBO,1939-01-01,Masculin,Marié,0.0,Seguela,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
1,012856Q15,012856Q,KHISSY BEYNIOUAH FULBERT,1939-01-01,Masculin,Célibataire,0.0,Bouaké,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
2,013454N15,013454N,VAHA TOMAS GNOMBLEI HENRI,1924-01-01,Masculin,Marié,0.0,Guiglo,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
3,027861L25,027861L,CAPET YATO MATHIEU,1943-03-01,Masculin,Marié,0.0,Abidjan,A affecter,"Min. Affaires Etrangères, de l'Intégration Afr...",...,a affecter,1800,"min affaires etrangeres, de l integration afri...",25,,<NA>,,<NA>,en activite,10
4,039659M38,039659M,COULIBALY YOUSSOUF,1945-12-01,Masculin,Marié,2.0,Abidjan,Dir Personnel Police Nationale,Min. de l'Intérieur et de la Sécurité,...,dir personnel police nationale,3813,min de l interieur et de la securite,38,commissaire divis 1er ech,LW,commissaire de police,P1ZC,retraite pour limite age,60


In [3]:
panel_solde_df.columns 

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'CODE_ORGANISME', 'CLASSE/ECHELON_NORM', 'CODE_CLASSE/ECHELON',
       'EMPLOI_NORM', 'CODE_EMPLOI', 'SITUATION_NORM', 'CODE_SITUATION'],
      dtype='object')

In [4]:
# Étape 1 : Nettoyer les données et extraire les valeurs uniques
unique_grades = panel_solde_df["GRADE"].dropna()           # Supprime les NaN
unique_grades = unique_grades[unique_grades != ""]         # Supprime les chaînes vides
unique_grades = unique_grades.unique().tolist()            # Liste des valeurs uniques

# Étape 2 : Créer un DataFrame à partir de la liste
grades_df = pd.DataFrame(unique_grades, columns=["GRADE"])

# Étape 3 : Enregistrer dans un fichier Excel
grades_df.to_excel("unique_grades.xlsx", index=False)

In [5]:
# Étape 1 : Nettoyer les données et extraire les valeurs uniques
unique_grade_2 = panel_solde_df["GRADE 2"].dropna()           # Supprime les NaN
unique_grade_2 = unique_grade_2[unique_grade_2 != ""]         # Supprime les chaînes vides
unique_grade_2 = unique_grade_2.unique().tolist()            # Liste des valeurs uniques

# Étape 2 : Créer un DataFrame à partir de la liste
grade_2_df = pd.DataFrame(unique_grade_2, columns=["GRADE 2"])

# Étape 3 : Enregistrer dans un fichier Excel
grade_2_df.to_excel("unique_grade_2.xlsx", index=False)

# FONCTIONS NECESSAIRES

### CREATION DE LA VARIABLE CATEGORIE 1 ET 2 

In [6]:
def build_categories(
    df: pd.DataFrame,
    grade_col="GRADE",
    grade2_col="GRADE 2",
    compute_categorie2=True,     # calcule CATEGORIE_2 depuis GRADE 2 si la colonne existe
    overwrite_categorie2=True    # réécrit CATEGORIE_2 même si déjà présente
):
    """
    - CATEGORIE_1 depuis GRADE (texte):
        'Non Fonctionnaire' si le texte contient 'non fonctionnaire' (insensible à la casse/espaces)
        sinon extrait la lettre A-D depuis 'Catégorie A7', 'Catégorie B', etc.
        NaN -> 'Non Fonctionnaire'
    - CATEGORIE_2 depuis GRADE 2 (lettres A-D):
        'A7' -> 'A', 'B' -> 'B', autres -> NaN -> 'Non Fonctionnaire'
    - Les 2 colonnes sont typées 'category' avec les mêmes modalités ordonnées.

    Retourne: df (copie) avec CATEGORIE_1 (et CATEGORIE_2 si demandé).
    """

    out = df.copy()
    cat_order = ["Non Fonctionnaire", "A", "B", "C", "D"]

    # ---------- helpers ----------
    def _cat_from_grade2_letter(val):
        """ Cette fonction transforme un GRADE 2 en catégorie """
        if pd.isna(val):
            return pd.NA  # si valeur manquante, on retourne pd.NA.
        s = str(val).strip().upper()  # on convertit en chaîne, on enlève les espaces et on met en majuscules
        if re.fullmatch(r"[ABCD]\d+", s):   # A7, B3, C1, D2...
            return s[0]
        if re.fullmatch(r"[ABCD]", s):      # A, B, C, D
            return s
        return pd.NA

    def _parse_cat_from_grade_text(val):
        if pd.isna(val):
            return pd.NA
        t = str(val)
        # ex.: "Non   fonctionnaire", "non-fonctionnaire"
        if re.search(r"\bnon\s*fonctionnaire\b", t, flags=re.I):
            return "Non Fonctionnaire"
        # ex.: "Catégorie A", "Categorie B7", "catégorie c titulaire"
        m = re.search(r"Cat[ée]gorie\s+([A-D])(?:\s*\d+)?", t, flags=re.I)
        if m:
            return m.group(1).upper()
        return pd.NA

    # ---------- CATEGORIE_1 depuis GRADE ----------
    if grade_col not in out.columns:
        raise KeyError(f"Colonne '{grade_col}' absente.")
    out["CATEGORIE_1"] = out[grade_col].apply(_parse_cat_from_grade_text)
    out["CATEGORIE_1"] = out["CATEGORIE_1"].fillna("Non Fonctionnaire") # Remplace les NaN par "Non Fonctionnaire"
    out["CATEGORIE_1"] = pd.Categorical(out["CATEGORIE_1"], categories=cat_order, ordered=True) #Transforme la colonne en type pd.Categorical avec l’ordre défini (cat_order)

    # ---------- (optionnel) CATEGORIE_2 depuis GRADE 2 ----------
    if compute_categorie2 and (grade2_col in out.columns):
        if overwrite_categorie2 or ("CATEGORIE_2" not in out.columns):
            cat2_raw = out[grade2_col].apply(_cat_from_grade2_letter)
            cat2 = cat2_raw.astype(object)
            cat2[pd.isna(cat2)] = "Non Fonctionnaire" # Remplace les NaN par "Non Fonctionnaire"
            out["CATEGORIE_2"] = pd.Categorical(cat2, categories=cat_order, ordered=True) # #Transforme la colonne en type pd.Categorical avec l’ordre défini (cat_order)

    return out 

### CREATION DE LA VARIABLE STATUT

In [11]:
def build_statut_from_cats(
    df: pd.DataFrame,
    emploi_col: str = "EMPLOI",
    cat1_col: str | None = None,                    # si None, on cherche "CATEGORIE_1" puis "CATEGORIE 1"
    cat1_candidates: tuple[str, str] = ("CATEGORIE_1", "CATEGORIE 1"),
    cat2_col: str | None = None,                    # si None, on cherche "CATEGORIE_2" puis "CATEGORIE 2"
    cat2_candidates: tuple[str, str] = ("CATEGORIE_2", "CATEGORIE 2"),
    label_cas: str = "Cas particulier",
    return_tables: bool = True
):
    """
    Règles (avec CATEGORIE_1 OU CATEGORIE_2):
      1) Si EMPLOI ≠ vide ET [cat == 'Non Fonctionnaire' pour l'une des deux]  -> STATUT = Cas particulier
      2) Si EMPLOI ≠ vide ET [cat ∈ {A,B,C,D} pour l'une des deux]              -> STATUT = Fonctionnaire
      3) Si EMPLOI = vide  ET [cat ∈ {A,B,C,D} pour l'une des deux]             -> STATUT = Cas particulier
      4) Si EMPLOI = vide  ET [cat == 'Non Fonctionnaire' pour l'une des deux]  -> STATUT = Non Fonctionnaire
      (sinon par défaut -> 'Non Fonctionnaire')

    Effets:
      - Ajoute/écrase la colonne STATUT (dtype 'category' ordonné).
      - Optionnellement retourne un report avec des tableaux de contrôle.

    Retour:
      - (df_out, report) si return_tables=True, sinon df_out
    """
    out = df.copy()

    # --- 0) Trouver les colonnes CATEGORIE_1 et CATEGORIE_2 ---
    def _pick_col(cand_list):
        for c in cand_list:
            if c in out.columns:
                return c
        return None

    if cat1_col is None:
        cat1_col = _pick_col(cat1_candidates)
    if cat2_col is None:
        cat2_col = _pick_col(cat2_candidates)

    if cat1_col is None or cat1_col not in out.columns:
        raise KeyError("Colonne 'CATEGORIE_1' (ou 'CATEGORIE 1') introuvable.")
    if cat2_col is None or cat2_col not in out.columns:
        raise KeyError("Colonne 'CATEGORIE_2' (ou 'CATEGORIE 2') introuvable.")
    if emploi_col not in out.columns:
        raise KeyError(f"Colonne '{emploi_col}' introuvable.")

    # --- 1) EMPLOI renseigné ? ---
    emp_norm = out[emploi_col].fillna("").astype(str).str.strip()
    has_emploi = emp_norm.ne("")

    # --- 2) Normalisation catégories ---
    def _norm_ascii_lower(x):
        if pd.isna(x):
            return ""
        s = str(x).strip()
        s = unicodedata.normalize("NFKD", s)
        s = s.encode("ascii", "ignore").decode("ascii")
        s = re.sub(r"\s+", " ", s).strip().lower()
        return s

    c1_norm = out[cat1_col].map(_norm_ascii_lower)
    c2_norm = out[cat2_col].map(_norm_ascii_lower)

    # Ex: "non   fonctionnaire" -> match
    c1_is_nf = c1_norm.str.contains(r"\bnon\s*fonctionnaire\b", na=False)
    c2_is_nf = c2_norm.str.contains(r"\bnon\s*fonctionnaire\b", na=False)

    # Ex: "a", "b", "c", "d"
    c1_is_abcd = c1_norm.str.fullmatch(r"[abcd]", na=False)
    c2_is_abcd = c2_norm.str.fullmatch(r"[abcd]", na=False)

    is_nf_any   = c1_is_nf | c2_is_nf
    is_abcd_any = c1_is_abcd | c2_is_abcd

    # --- 3) STATUT (ordre des règles important) ---
    # 1) EMPLOI ≠ vide et Non Fonctionnaire -> Cas particulier
    # 2) EMPLOI ≠ vide et A-D -> Fonctionnaire
    # 3) EMPLOI = vide  et A-D -> Cas particulier
    # 4) EMPLOI = vide  et Non Fonctionnaire -> Non Fonctionnaire
    # default -> Non Fonctionnaire
    conditions = [
        has_emploi & is_nf_any,
        has_emploi & is_abcd_any,
        (~has_emploi) & is_abcd_any,
        (~has_emploi) & is_nf_any
    ]
    choices = [
        label_cas,
        "Fonctionnaire",
        label_cas,
        "Non Fonctionnaire"
    ]

    statut = np.select(conditions, choices, default="Non Fonctionnaire")

    out["STATUT"] = pd.Categorical(
        statut,
        categories=["Non Fonctionnaire", "Fonctionnaire", label_cas],
        ordered=True
    )

    if return_tables:
        rep = {
            "repartition_statut": out["STATUT"].value_counts().reindex(
                ["Non Fonctionnaire", "Fonctionnaire", label_cas], fill_value=0
            ),
            "nb_emploi_vide": int((emp_norm == "").sum()),
            "nb_cas_particulier_emploi_nf": int((has_emploi & is_nf_any).sum()),
            "nb_cas_particulier_sans_emploi_abcd": int((~has_emploi & is_abcd_any).sum()),
            "controle_cat_any": pd.crosstab(
                pd.Series(np.where(is_nf_any, "Non Fonctionnaire", np.where(is_abcd_any, "A-D", "Autre/NaN")),
                          name="Catégorie (any)"),
                out["STATUT"]
            )
        }
        return out, rep

    return out


### CREATION DE LA VARIABLE STATUT DEFINITIF 

Créer une variable Statut_def qui est  fonctionnaire (via l'utilisation de la variable STATUT) si pour un même matricule il est fonctionnaire et non fonctionnaire pour l'organisme (utiliser le CODE ORGANISME QUI PEUT VARIER DANS LE TEMPS)  sur toute la periode collecte en utilisant la variable DATE_COLLECTE  et Statut_def qui est  non fonctionnaire si  pour un même matricule il non fonctionnaire  pour l'organisme sur toute la periode de collecte 

In [8]:
def build_statut_def(
    df: pd.DataFrame,
    statut_col: str = "STATUT",
    matricule_col: str = "MATRICULE",
    org_col: str | None = None,                # si None: détection auto
    composite_col: str = "MATRICULE||CODE_ORGANISME",
    treat_cas_as_functionnaire: bool = True,   # "Cas particulier" compte comme Fonctionnaire
    output_col: str = "Statut_def",
    return_report: bool = True
):
    """
    Règle:
      - Pour chaque couple (MATRICULE, ORGANISME), si le statut est au moins une fois
        'Fonctionnaire' (et 'Cas particulier' si treat_cas_as_functionnaire=True),
        alors Statut_def = 'Fonctionnaire' pour TOUTES les lignes de ce couple.
        Sinon, 'Non Fonctionnaire'.

    Colonnes attendues: STATUT + clé de groupage (voir détection auto ci-dessous).
    Détection auto de l'organisme (dans cet ordre): 
      - org_col (si fourni) 
      - 'MATRICULE||CODE_ORGANISME' 
      - ('MATRICULE', 'ID_ORGANISME')
      - ('MATRICULE', 'CODE_ORGANISME')
    Retour: (df_out, report) si return_report=True, sinon df_out.
    """
    out = df.copy()

    # ---- 0) Trouver la clé de groupage ----
    if org_col is not None and org_col in out.columns:
        key_cols = [matricule_col, org_col]
    elif composite_col in out.columns:
        key_cols = [composite_col]
    elif {matricule_col, "ID_ORGANISME"}.issubset(out.columns):
        key_cols = [matricule_col, "ID_ORGANISME"]
    elif {matricule_col, "CODE_ORGANISME"}.issubset(out.columns):
        key_cols = [matricule_col, "CODE_ORGANISME"]
    else:
        raise KeyError(
            "Clé introuvable : fournissez org_col OU une des combinaisons "
            f"['{composite_col}'] ou ('{matricule_col}','ID_ORGANISME') ou "
            f"('{matricule_col}','CODE_ORGANISME')."
        )

    if statut_col not in out.columns:
        raise KeyError(f"Colonne '{statut_col}' introuvable.")

    # ---- 1) Normalisation du STATUT -> est_fonctionnaire ----
    stat = out[statut_col].astype(str).str.strip().str.casefold()
    func_values = {"fonctionnaire"}
    if treat_cas_as_functionnaire:
        func_values.add("cas particulier")
    est_fonctionnaire = stat.isin(func_values)

    # ---- 2) Agrégation par couple -> "au moins une fois fonctionnaire ?" ----
    any_fonct_by_pair = est_fonctionnaire.groupby([out[c] for c in key_cols]).transform("any")

    # ---- 3) Statut_def par ligne (catégorielle ordonnée) ----
    out[output_col] = np.where(any_fonct_by_pair, "Fonctionnaire", "Non Fonctionnaire")
    out[output_col] = pd.Categorical(out[output_col],
                                     categories=["Non Fonctionnaire", "Fonctionnaire"],
                                     ordered=True)

    if return_report:
        # résumé par couple
        pair_df = pd.DataFrame({ "_is_f": est_fonctionnaire })
        pair_df = pair_df.join(out[key_cols])
        pair_summary = (
            pair_df.groupby(key_cols, dropna=False)["_is_f"]
                   .agg(any_f="any", all_nf=lambda s: (~s).all())
                   .reset_index()
        )
        rep = {
            "repartition_lignes": out[output_col].value_counts()
                                   .reindex(["Non Fonctionnaire","Fonctionnaire"], fill_value=0),
            "nb_couples_total": int(pair_summary.shape[0]),
            "nb_couples_avec_>=1_fonctionnaire": int(pair_summary["any_f"].sum()),
            "nb_couples_toujours_non_fonctionnaire": int(pair_summary["all_nf"].sum()),
        }
        return out, rep

    return out



### CREATION DE LA VARIABLE AGE IMPUTE


In [9]:
def build_age_valide(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    birth_col="DATE NAISSANCE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE",                # colonne sexe brute
    sex_valid_col="SEXE_VALIDE",   # colonne sexe imputée
    strategy="recent",             # "recent" (plus récente) ou "mode" (valeur la plus fréquente)
    age_min=18, age_max=70,
    do_impute_age=True,            # imputer AGE_VALIDE -> AGE_IMPUTE par médiane
    return_tables=True,            # retourner tables sexe (avant/après)
    debug=False
):
    """
    Nettoie les dates de naissance, crée DATE_NAISSANCE_CORRIGEE (plus récente ou mode),
    impute SEXE manquant par le mode (ANNEE_COLLECTE × MOIS_COLLECTE),
    calcule AGE à la date de collecte + AGE_VALIDE borné [age_min, age_max],
    et (option) impute AGE_IMPUTE par médiane (ANNEE×MOIS×SEXE_VALIDE).

    Retour:
      - (df_enrichi, report) si return_tables=True
      - df_enrichi sinon
    """
    df = df.copy()

    # --------- utilitaires ---------
    def to_datetime_mixed(s):
        # pandas >= 2.0: format="mixed"; fallback pour <2.0
        try:
            return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=True)
        except TypeError:
            return pd.to_datetime(s, errors="coerce", dayfirst=True, infer_datetime_format=True)

    def clean_birthdates(series):
        s = series.copy()
        s_str = s.astype(str).str.strip().str.lower()

        time_only = s_str.str.fullmatch(r"\d{1,2}:\d{2}(:\d{2})?")
        zero_date = s_str.str.fullmatch(r"(0{1,2}[/\-]0{1,2}[/\-]0{2,4}|0000-00-00)")
        empties   = s_str.isin(["", "nan", "nat", "none", "nul", "null"])

        s = s.mask(time_only | zero_date | empties, np.nan)
        dt = to_datetime_mixed(s)

        # numéros Excel (ex: "44927")
        serial = pd.to_numeric(s_str, errors="coerce")
        serial_mask = dt.isna() & serial.notna() & serial.between(1, 60000)
        if serial_mask.any():
            dt_serial = pd.to_datetime(serial[serial_mask], unit="D",
                                       origin="1899-12-30", errors="coerce")
            dt.loc[serial_mask] = dt_serial
        return dt

    # --------- 0) DATE_COLLECTE -> datetime + dérivées ---------
    if collect_col not in df.columns:
        raise KeyError(f"Colonne '{collect_col}' absente.")
    df[collect_col] = to_datetime_mixed(df[collect_col])
    if "ANNEE_COLLECTE" not in df.columns:
        df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    if "MOIS_COLLECTE" not in df.columns:
        df["MOIS_COLLECTE"] = df[collect_col].dt.month

    # --------- 1) Nettoyage DATE NAISSANCE + nbre distinct ---------
    if birth_col not in df.columns:
        raise KeyError(f"Colonne '{birth_col}' absente.")
    df["DATE_NAISSANCE_CLEAN"] = clean_birthdates(df[birth_col])

    if matricule_col not in df.columns:
        raise KeyError(f"Colonne '{matricule_col}' absente.")
    nbre_date = df.groupby(matricule_col)["DATE_NAISSANCE_CLEAN"].nunique(dropna=True)
    df["nbre_date_naissances"] = df[matricule_col].map(nbre_date)

    # --------- 2) Choix DATE_NAISSANCE_CORRIGEE ---------
    if strategy == "recent":
        tmp = (df[[matricule_col, collect_col, "DATE_NAISSANCE_CLEAN"]]
               .sort_values([matricule_col, collect_col], ascending=[True, False])
               .drop_duplicates(subset=[matricule_col], keep="first"))
        date_corr = (tmp
                     .rename(columns={"DATE_NAISSANCE_CLEAN": "DATE_NAISSANCE_CORRIGEE"})
                     [[matricule_col, "DATE_NAISSANCE_CORRIGEE"]])
    elif strategy == "mode":
        mode_df = (df.dropna(subset=["DATE_NAISSANCE_CLEAN"])
                     .groupby(matricule_col)["DATE_NAISSANCE_CLEAN"]
                     .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else pd.NaT)
                     .reset_index(name="DATE_NAISSANCE_CORRIGEE"))
        date_corr = mode_df
    else:
        raise ValueError("strategy doit être 'recent' ou 'mode'.")

    if debug:
        print("[DEBUG] date_corr shape:", date_corr.shape)
        print("[DEBUG] date_corr cols:", list(date_corr.columns)[:5])

    # 🔒 Patch A: éviter les suffixes si la colonne existe déjà
    if "DATE_NAISSANCE_CORRIGEE" in df.columns:
        df.drop(columns=["DATE_NAISSANCE_CORRIGEE"], inplace=True)

    # Merge many->one garanti
    df = df.merge(date_corr, on=matricule_col, how="left", validate="many_to_one")

    # Conversion sûre en datetime
    df["DATE_NAISSANCE_CORRIGEE"] = pd.to_datetime(df["DATE_NAISSANCE_CORRIGEE"], errors="coerce")

    # Composantes Y/M/D
    df["ANNEE_NAISSANCE_CORRIGEE"] = df["DATE_NAISSANCE_CORRIGEE"].dt.year
    df["MOIS_NAISSANCE_CORRIGEE"]  = df["DATE_NAISSANCE_CORRIGEE"].dt.month
    df["JOUR_NAISSANCE_CORRIGEE"]  = df["DATE_NAISSANCE_CORRIGEE"].dt.day

    # --------- 3) Imputation SEXE par le mode (ANNEE×MOIS) ---------
    tab_sexe_avant = df[sex_col].value_counts(dropna=False).sort_index()
    tab_sexe_avant_pct = df[sex_col].value_counts(normalize=True, dropna=False).sort_index()

    df[sex_valid_col] = df[sex_col]

    mode_sexe_groupes = (
        df.dropna(subset=[sex_col])
          .groupby(["ANNEE_COLLECTE", "MOIS_COLLECTE"])[sex_col]
          .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )

    def _imputer_sexe_par_mode(row):
        if pd.isna(row[sex_valid_col]):
            return mode_sexe_groupes.get((row["ANNEE_COLLECTE"], row["MOIS_COLLECTE"]), np.nan)
        else:
            return row[sex_valid_col]

    df[sex_valid_col] = df.apply(_imputer_sexe_par_mode, axis=1)

    tab_sexe_apres = df[sex_valid_col].value_counts(dropna=False).sort_index()
    tab_sexe_apres_pct = df[sex_valid_col].value_counts(normalize=True, dropna=False).sort_index()

    crosstab_sexe = pd.crosstab(
        index=df[sex_col],
        columns=df[sex_valid_col],
        dropna=False,
        margins=True,
        margins_name="Total"
    )

    # --------- 4) AGE & AGE_VALIDE ---------
    birth = df["DATE_NAISSANCE_CORRIGEE"]
    ref   = df[collect_col]

    age = pd.Series(pd.NA, index=df.index, dtype="Float64")
    mask = birth.notna() & ref.notna()
    if mask.any():
        ydiff = ref[mask].dt.year - birth[mask].dt.year
        before_bday = (ref[mask].dt.month < birth[mask].dt.month) | (
            (ref[mask].dt.month == birth[mask].dt.month) & (ref[mask].dt.day < birth[mask].dt.day)
        )
        age.loc[mask] = (ydiff - before_bday.astype(int)).astype("Float64")

    df["AGE"] = age
    df["AGE_VALIDE"] = df["AGE"].where((df["AGE"].ge(age_min)) & (df["AGE"].le(age_max)))

    # --------- 5) Imputation AGE par médiane (option) ---------
    if do_impute_age:
        med = (df.groupby(["ANNEE_COLLECTE", "MOIS_COLLECTE", sex_valid_col])["AGE_VALIDE"]
                 .transform("median"))
        df["AGE_IMPUTE"] = df["AGE_VALIDE"].where(df["AGE_VALIDE"].notna(), med).round(0)

    if return_tables:
        report = {
            "tab_sexe_avant": tab_sexe_avant,
            "tab_sexe_avant_pct": (tab_sexe_avant_pct * 100).round(2),
            "tab_sexe_apres": tab_sexe_apres,
            "tab_sexe_apres_pct": (tab_sexe_apres_pct * 100).round(2),
            "crosstab_sexe": crosstab_sexe,
        }
        return df, report
    else:
        return df


### CREATION DE LA VARIABLE AGE DE PRISE DE SERVICE 


In [10]:
def build_age_service(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    start_col_raw="PRISE DE SERVICE",          # colonne brute (texte/mixtes)
    start_col_corrected="PRISE_DE_SERVICE_CORRIGEE",
    birth_col="DATE_NAISSANCE_CORRIGEE",
    collect_col="DATE_COLLECTE",
    sex_valid_col="SEXE_VALIDE",
    year_collect_col="ANNEE_COLLECTE",
    month_collect_col="MOIS_COLLECTE",
    recompute_corrected=True,                  # recalcule PRISE_DE_SERVICE_CORRIGEE depuis la collecte la + récente
    age_min=18,
    age_max=40,
    days_per_year=365,                         # fidèle à ton // 365 ; mettre 365.25 si besoin
    return_tables=True,
    sample_anomalies=10                        # nb de lignes à montrer pour l’aperçu anomalies
):
    """
    Étapes intégrées :
      1) Nettoie PRISE DE SERVICE (formats mixtes, placeholders, numéros Excel).
      2) Diagnostic : nb de dates uniques par MATRICULE, liste des matricules 'à problème'.
      3) Calcule PRISE_DE_SERVICE_CORRIGEE = valeur de PRISE DE SERVICE (clean)
         observée à la collecte la plus récente par MATRICULE.
      4) Calcule AGE_AU_SERVICE, AGE_AU_SERVICE_VALIDE (bornage [18,40]).
      5) Impute AGE_AU_SERVICE_IMPUTE par médiane (ANNEE_COLLECTE × MOIS_COLLECTE × SEXE_VALIDE si dispo).
      6) Retourne df enrichi + report (diagnostics et tableaux).

    Retour:
      - (df_enrichi, report) si return_tables=True
      - df_enrichi sinon
    """
    df = df.copy()

    # ---------- utilitaires ----------
    def _to_datetime_mixed(s):
        try:
            return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=True)
        except TypeError:  # pandas < 2.0
            return pd.to_datetime(s, errors="coerce", dayfirst=True, infer_datetime_format=True)

    def _clean_dates_generic(series):
        """Nettoie dates: supprime heures seules, '0000-00-00', vides; gère numéros Excel."""
        s = series.copy()
        s_str = s.astype(str).str.strip().str.lower()

        time_only = s_str.str.fullmatch(r"\d{1,2}:\d{2}(:\d{2})?")
        zero_date = s_str.str.fullmatch(r"(0{1,2}[/\-]0{1,2}[/\-]0{2,4}|0000-00-00)")
        empties   = s_str.isin(["", "nan", "nat", "none", "nul", "null"])

        s = s.mask(time_only | zero_date | empties, np.nan)
        dt = _to_datetime_mixed(s)

        # Numéros Excel éventuels (ex: "44927")
        serial = pd.to_numeric(s_str, errors="coerce")
        serial_mask = dt.isna() & serial.notna() & serial.between(1, 60000)
        if serial_mask.any():
            dt_serial = pd.to_datetime(serial[serial_mask], unit="D",
                                       origin="1899-12-30", errors="coerce")
            dt.loc[serial_mask] = dt_serial
        return dt

    # ---------- 0) Sécuriser DATE_COLLECTE et dérivées ----------
    if collect_col not in df.columns:
        raise KeyError(f"Colonne '{collect_col}' absente.")
    df[collect_col] = _to_datetime_mixed(df[collect_col])
    if year_collect_col not in df.columns:
        df[year_collect_col] = df[collect_col].dt.year
    if month_collect_col not in df.columns:
        df[month_collect_col] = df[collect_col].dt.month

    # ---------- 1) Nettoyage PRISE DE SERVICE ----------
    if start_col_raw not in df.columns:
        raise KeyError(f"Colonne '{start_col_raw}' absente.")
    df["PRISE_DE_SERVICE_CLEAN"] = _clean_dates_generic(df[start_col_raw])

    # ---------- 2) Diagnostics: nb de dates uniques / matricule ----------
    if matricule_col not in df.columns:
        raise KeyError(f"Colonne '{matricule_col}' absente.")
    nb_dates_par_matricule = df.groupby(matricule_col)["PRISE_DE_SERVICE_CLEAN"].nunique(dropna=True)
    matricules_problemes = nb_dates_par_matricule[nb_dates_par_matricule > 1].index.tolist()

    anomalies_dates = (
        df[df[matricule_col].isin(matricules_problemes)]
        [[matricule_col, collect_col, "PRISE_DE_SERVICE_CLEAN"]]
        .sort_values([matricule_col, collect_col])
    )

    # ---------- 3) PRISE_DE_SERVICE_CORRIGEE = valeur à la collecte la + récente ----------
    if recompute_corrected or (start_col_corrected not in df.columns):
        # Trier (matricule asc, collecte desc), garder la première (la plus récente)
        panel_sorted = df.sort_values([matricule_col, collect_col], ascending=[True, False])
        ps_corr = (
            panel_sorted
            .drop_duplicates(subset=[matricule_col], keep="first")
            [[matricule_col, "PRISE_DE_SERVICE_CLEAN"]]
            .rename(columns={"PRISE_DE_SERVICE_CLEAN": start_col_corrected})
        )
        # Éviter les suffixes si la colonne existe déjà
        if start_col_corrected in df.columns:
            df.drop(columns=[start_col_corrected], inplace=True)
        df = df.merge(ps_corr, on=matricule_col, how="left", validate="many_to_one")

    # Conversion sûre
    df[start_col_corrected] = _to_datetime_mixed(df[start_col_corrected])

    # Tests d’unicité après correction
    test_unicite = df.groupby(matricule_col)[start_col_corrected].nunique(dropna=True)
    test_unicite_max = int(test_unicite.max()) if len(test_unicite) else 0
    nb_indiv_multi = int((test_unicite > 1).sum())

    # ---------- 4) AGE_AU_SERVICE & AGE_AU_SERVICE_VALIDE ----------
    if birth_col not in df.columns:
        raise KeyError(f"Colonne '{birth_col}' absente.")
    if not np.issubdtype(df[birth_col].dtype, np.datetime64):
        df[birth_col] = _to_datetime_mixed(df[birth_col])

    mask = df[birth_col].notna() & df[start_col_corrected].notna()
    age_service = pd.Series(pd.NA, index=df.index, dtype="Float64")
    if mask.any():
        delta_days = (df.loc[mask, start_col_corrected] - df.loc[mask, birth_col]).dt.days
        age_service.loc[mask] = (delta_days // days_per_year).astype("Float64")

    df["AGE_AU_SERVICE"] = age_service
    df["AGE_AU_SERVICE_VALIDE"] = df["AGE_AU_SERVICE"].where(
        (df["AGE_AU_SERVICE"].ge(age_min)) & (df["AGE_AU_SERVICE"].le(age_max))
    )

    # ---------- 5) Imputation par médiane (ANNEE×MOIS×SEXE_VALIDE si dispo) ----------
    group_keys = [year_collect_col, month_collect_col]
    if sex_valid_col in df.columns:
        group_keys.append(sex_valid_col)

    med_group = df.groupby(group_keys)["AGE_AU_SERVICE_VALIDE"].transform("median")
    df["AGE_AU_SERVICE_IMPUTE"] = df["AGE_AU_SERVICE_VALIDE"].where(
        df["AGE_AU_SERVICE_VALIDE"].notna(), med_group
    ).round(0)

    if return_tables:
        # Tables validé
        tab_valide = df["AGE_AU_SERVICE_VALIDE"].value_counts(dropna=False).sort_index()
        tab_valide_pct = df["AGE_AU_SERVICE_VALIDE"].value_counts(normalize=True, dropna=False).sort_index()
        # Tables imputé
        tab_impute = df["AGE_AU_SERVICE_IMPUTE"].value_counts(dropna=False).sort_index()
        tab_impute_pct = df["AGE_AU_SERVICE_IMPUTE"].value_counts(normalize=True, dropna=False).sort_index()

        report = {
            "n_matricules_multi_service": len(matricules_problemes),
            "anomalies_dates_head": anomalies_dates.head(sample_anomalies),
            "test_unicite_max": test_unicite_max,
            "nb_indiv_plusieurs_dates_apres_correction": nb_indiv_multi,
            "tab_age_service_valide": pd.DataFrame({
                "Effectif": tab_valide,
                "Pourcentage (%)": (tab_valide_pct * 100).round(2)
            }),
            "tab_age_service_impute": pd.DataFrame({
                "Effectif": tab_impute,
                "Pourcentage (%)": (tab_impute_pct * 100).round(2)
            }),
        }
        return df, report

    return df


### CREATION DE LA VARIBALE ANCIENNTE DANS L'ORGANISME ET DEPUIS LA PRISE DE FONCTION

**ANCIENTE DANS LA PRISE DE FONCTION = AGE IMPUTER DE LA COLLECTE - AGE DE LA PRISE DE SERVICE**

**DATE_ENTREE_ORGANISME_ADJ = max(min(DATE_COLLECTE), PRISE_DE_SERVICE_CORRIGEE)**

**ANCIENNETE DANS L'ORGANISME = MAX(DATE DE COLLECTE DANS l'ORGANISME) - DATE_ENTREE_ORGANISME_ADJ**

In [26]:
import pandas as pd
import numpy as np

def build_anciennete(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    org_col=None,                             # auto: ID_ORGANISME > CODE_ORGANISME > MATRICULE||CODE_ORGANISME
    collect_col="DATE_COLLECTE",
    age_impute_col="AGE_IMPUTE",
    age_service_impute_col="AGE_AU_SERVICE_IMPUTE",
    start_service_col="PRISE_DE_SERVICE_CORRIGEE",   # pour contraindre l'entrée org
    days_per_year=365,
    adjust_entry_with_service=True,           # impose DATE_ENTREE_ORGANISME_ADJ >= PRISE_DE_SERVICE_CORRIGEE
    clip_to_function=True,                    # borne l'ancienneté org <= ancienneté depuis la fonction
    broadcast_latest_org=True,                # diffuse la dernière ancienneté org sur tout le couple
    broadcast_latest_fonction=False,          # optionnel pour l'ancienneté depuis la fonction
    do_impute_org=False,                      # impute médiane par (ANNEE, MOIS[, SEXE_VALIDE])
    keep_raw=True,                            # conserve ANCIENNETE_ORGANISME_RAW (avant ajustement/clip)
    return_tables=True
):
    """
    Ajoute :
      - ANCIENNETE_DEPUIS_FONCTION = AGE_IMPUTE - AGE_AU_SERVICE_IMPUTE (négatifs -> NaN)
      - DATE_ENTREE_ORGANISME_RAW = min(DATE_COLLECTE) par (MATRICULE, ORGANISME)
      - DATE_ENTREE_ORGANISME_ADJ = max(RAW, PRISE_DE_SERVICE_CORRIGEE) si dispo
      - ANCIENNETE_ORGANISME = floor((DATE_COLLECTE - DATE_ENTREE_ORGANISME_ADJ)/365)
      - (option) ANCIENNETE_ORGANISME_RAW, ANCIENNETE_ORGANISME_IMPUTE
      - (option) ANCIENNETE_ORGANISME_MAJ (valeur à la DERNIÈRE collecte, diffusée partout)
      - (option) ANCIENNETE_DEPUIS_FONCTION_MAJ
      - Alias pratique : DATE_ENTREE_ORGANISME = DATE_ENTREE_ORGANISME_ADJ
    """
    df = df.copy()

    # ---- utils ----
    def _to_dt(s):
        try:
            return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=True)
        except TypeError:
            return pd.to_datetime(s, errors="coerce", dayfirst=True, infer_datetime_format=True)

    def _years_floor(d1, d0):
        mask = d1.notna() & d0.notna()
        out = pd.Series(pd.NA, index=d1.index, dtype="Float64")
        if mask.any():
            out.loc[mask] = ((d1[mask] - d0[mask]).dt.days // days_per_year).astype("Float64")
        return out.where(out.isna() | (out >= 0))  # pas de négatifs

    # ---- colonnes clés ----
    if collect_col not in df.columns:
        raise KeyError(f"Colonne '{collect_col}' absente.")
    df[collect_col] = _to_dt(df[collect_col])

    if org_col is None:
        for c in ["ID_ORGANISME", "CODE_ORGANISME", "MATRICULE||CODE_ORGANISME"]:
            if c in df.columns:
                org_col = c
                break
        if org_col is None:
            raise KeyError("Aucune colonne organisme trouvée (ID_ORGANISME / CODE_ORGANISME / MATRICULE||CODE_ORGANISME).")

    if "ANNEE_COLLECTE" not in df.columns:
        df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    if "MOIS_COLLECTE" not in df.columns:
        df["MOIS_COLLECTE"] = df[collect_col].dt.month

    # ---- 1) Ancienneté depuis la prise de fonction ----
    if age_impute_col not in df.columns or age_service_impute_col not in df.columns:
        raise KeyError("AGE_IMPUTE et/ou AGE_AU_SERVICE_IMPUTE manquent. Exécute d'abord build_age_valide et build_age_service.")
    anc_fct = df[age_impute_col].astype("Float64") - df[age_service_impute_col].astype("Float64")
    anc_fct = anc_fct.where(anc_fct.isna() | (anc_fct >= 0))  # négatifs -> NaN
    df["ANCIENNETE_DEPUIS_FONCTION"] = anc_fct.round(0)

    # ---- 2) DATE_ENTREE_ORGANISME_RAW = min(DATE_COLLECTE) ----
    df["DATE_ENTREE_ORGANISME_RAW"] = (
        df.groupby([matricule_col, org_col], dropna=False)[collect_col]
          .transform("min")
    )

    # ---- 3) Date d'entrée AJUSTÉE (>= prise de service si dispo) ----
    if start_service_col in df.columns and adjust_entry_with_service:
        df[start_service_col] = _to_dt(df[start_service_col])
        df["DATE_ENTREE_ORGANISME_ADJ"] = df["DATE_ENTREE_ORGANISME_RAW"].where(
            df[start_service_col].isna() | (df["DATE_ENTREE_ORGANISME_RAW"] >= df[start_service_col]),
            df[start_service_col]
        )
    else:
        df["DATE_ENTREE_ORGANISME_ADJ"] = df["DATE_ENTREE_ORGANISME_RAW"]

    # ---- 4) Ancienneté ORG (ajustée) ----
    if keep_raw:
        df["ANCIENNETE_ORGANISME_RAW"] = _years_floor(df[collect_col], df["DATE_ENTREE_ORGANISME_RAW"])
    df["ANCIENNETE_ORGANISME"] = _years_floor(df[collect_col], df["DATE_ENTREE_ORGANISME_ADJ"])

    # ---- 5) Clip à l'ancienneté depuis la fonction (sécurité) ----
    if clip_to_function:
        gt_mask = (
            df["ANCIENNETE_ORGANISME"].notna()
            & df["ANCIENNETE_DEPUIS_FONCTION"].notna()
            & (df["ANCIENNETE_ORGANISME"] > df["ANCIENNETE_DEPUIS_FONCTION"])
        )
        df.loc[gt_mask, "ANCIENNETE_ORGANISME"] = df.loc[gt_mask, "ANCIENNETE_DEPUIS_FONCTION"]

    # ---- 6) Alias pratique pour l'affichage ----
    df["DATE_ENTREE_ORGANISME"] = df["DATE_ENTREE_ORGANISME_ADJ"]

    # ---- 7) Diffuser la DERNIÈRE valeur au sein du couple (robuste) ----
    if broadcast_latest_org:
        base = df.dropna(subset=[collect_col]).sort_values([matricule_col, org_col, collect_col])
        last_rows = (
            base.groupby([matricule_col, org_col], as_index=False).tail(1)
                [[matricule_col, org_col, "ANCIENNETE_ORGANISME"]]
                .rename(columns={"ANCIENNETE_ORGANISME": "ANCIENNETE_ORGANISME_MAJ"})
        )
        # s'assurer de recréer proprement la colonne
        if "ANCIENNETE_ORGANISME_MAJ" in df.columns:
            df.drop(columns=["ANCIENNETE_ORGANISME_MAJ"], inplace=True)
        df = df.merge(last_rows, on=[matricule_col, org_col], how="left", validate="many_to_one")

    if broadcast_latest_fonction:
        base_f = df.dropna(subset=[collect_col]).sort_values([matricule_col, org_col, collect_col])
        last_rows_f = (
            base_f.groupby([matricule_col, org_col], as_index=False).tail(1)
                  [[matricule_col, org_col, "ANCIENNETE_DEPUIS_FONCTION"]]
                  .rename(columns={"ANCIENNETE_DEPUIS_FONCTION": "ANCIENNETE_DEPUIS_FONCTION_MAJ"})
        )
        if "ANCIENNETE_DEPUIS_FONCTION_MAJ" in df.columns:
            df.drop(columns=["ANCIENNETE_DEPUIS_FONCTION_MAJ"], inplace=True)
        df = df.merge(last_rows_f, on=[matricule_col, org_col], how="left", validate="many_to_one")

    # ---- 8) Imputation (optionnelle) de l'ancienneté org ----
    if do_impute_org:
        group_keys = ["ANNEE_COLLECTE", "MOIS_COLLECTE"]
        if "SEXE_VALIDE" in df.columns:
            group_keys.append("SEXE_VALIDE")
        med_org = df.groupby(group_keys)["ANCIENNETE_ORGANISME"].transform("median")
        df["ANCIENNETE_ORGANISME_IMPUTE"] = df["ANCIENNETE_ORGANISME"].where(
            df["ANCIENNETE_ORGANISME"].notna(), med_org
        ).round(0)

    # ---- 9) Report ----
    if return_tables:
        rep = {
            "nb_lignes_collecte_manquante": int(df[collect_col].isna().sum()),
            "tab_anc_org": df["ANCIENNETE_ORGANISME"].value_counts(dropna=False).sort_index(),
            "tab_anc_fonction": df["ANCIENNETE_DEPUIS_FONCTION"].value_counts(dropna=False).sort_index(),
            "tab_anc_org_maj": (
                df.get("ANCIENNETE_ORGANISME_MAJ", pd.Series(dtype="Float64"))
                  .value_counts(dropna=False).sort_index()
                if broadcast_latest_org else None
            ),
        }
        if keep_raw:
            rep["tab_anc_org_raw"] = df["ANCIENNETE_ORGANISME_RAW"].value_counts(dropna=False).sort_index()
        if do_impute_org:
            rep["tab_anc_org_impute"] = df["ANCIENNETE_ORGANISME_IMPUTE"].value_counts(dropna=False).sort_index()
        return df, rep

    return df


# AFFICHAGE DES DIFFERENTES VARIABLES CREES 



#### CATEGORIE 

In [13]:
panel_solde_df = build_categories(panel_solde_df, grade_col="GRADE", grade2_col="GRADE 2")
# Aperçu
panel_solde_df[["GRADE","CATEGORIE_1","GRADE 2","CATEGORIE_2"]].head()

,GRADE,CATEGORIE_1,GRADE 2,CATEGORIE_2
0,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
1,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
2,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
3,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
4,Fonctionnaire Catégorie A7,A,A7,A


In [14]:
# 📊 Créer un tableau croisé des effectifs
tableau_croise_effectifs = pd.crosstab(
    panel_solde_df["CATEGORIE_1"],   # Variable en ligne
    panel_solde_df["CATEGORIE_2"],   # Variable en colonne
    margins=True,        # Ajoute les totaux
    margins_name="Total" # Nom de la ligne/colonne total
)
print(tableau_croise_effectifs)

CATEGORIE_2        Non Fonctionnaire        A        B        C       D  \
CATEGORIE_1                                                               
Non Fonctionnaire             320356       43        0        0       0   
A                                  0  4797789        0        0       0   
B                                  0        0  6119667        0       0   
C                                  0        0        0  3961427       0   
D                                  0        0        0        0  428681   
Total                         320356  4797832  6119667  3961427  428681   

CATEGORIE_2           Total  
CATEGORIE_1                  
Non Fonctionnaire    320399  
A                   4797789  
B                   6119667  
C                   3961427  
D                    428681  
Total              15627963  


#### STATUT

In [16]:
panel_solde_df, rep = build_statut_from_cats(panel_solde_df, return_tables=True)

print(rep["repartition_statut"])
print("EMPLOI vide :", rep["nb_emploi_vide"])
print(
    "Cas particulier :",
    rep["nb_cas_particulier_emploi_nf"] + rep["nb_cas_particulier_sans_emploi_abcd"]
)

# aperçu (tu peux ajouter CATEGORIE_1 si utile)
panel_solde_df[["EMPLOI", "CATEGORIE_2","CATEGORIE_1", "STATUT"]].head()


STATUT
Non Fonctionnaire      315125
Fonctionnaire        15307347
Cas particulier          5491
Name: count, dtype: int64
EMPLOI vide : 315378
Cas particulier : 5491


,EMPLOI,CATEGORIE_2,CATEGORIE_1,STATUT
0,None,Non Fonctionnaire,Non Fonctionnaire,Non Fonctionnaire
1,None,Non Fonctionnaire,Non Fonctionnaire,Non Fonctionnaire
2,None,Non Fonctionnaire,Non Fonctionnaire,Non Fonctionnaire
3,None,Non Fonctionnaire,Non Fonctionnaire,Non Fonctionnaire
4,Commissaire de Police,A,A,Fonctionnaire


#### STATUT DEF

In [17]:
# Appliquer et récupérer le report
panel_solde_df, rep = build_statut_def(panel_solde_df, return_report=True)

print("Répartition (toutes lignes) :")
print(rep["repartition_lignes"])

# Aperçu
panel_solde_df[["MATRICULE","ID_ORGANISME","STATUT","Statut_def"]].head()


Répartition (toutes lignes) :
Statut_def
Non Fonctionnaire      309851
Fonctionnaire        15318112
Name: count, dtype: int64
Couples (Matricule x Organisme) - total : 308240
  avec >=1 Fonctionnaire : 294430
  toujours Non Fonctionnaire : 13810


,MATRICULE,ID_ORGANISME,STATUT,Statut_def
0,011242X,15,Non Fonctionnaire,Non Fonctionnaire
1,012856Q,15,Non Fonctionnaire,Non Fonctionnaire
2,013454N,15,Non Fonctionnaire,Non Fonctionnaire
3,027861L,25,Non Fonctionnaire,Non Fonctionnaire
4,039659M,38,Fonctionnaire,Fonctionnaire


In [18]:
# 📊 Créer un tableau croisé des effectifs
tableau_croise_effectifs = pd.crosstab(
    panel_solde_df["Statut_def"],   # Variable en ligne
    panel_solde_df["STATUT"],   # Variable en colonne
    margins=True,        # Ajoute les totaux
    margins_name="Total" # Nom de la ligne/colonne total
)
print(tableau_croise_effectifs)

STATUT             Non Fonctionnaire  Fonctionnaire  Cas particulier     Total
Statut_def                                                                    
Non Fonctionnaire             309851              0                0    309851
Fonctionnaire                   5527       15307354             5231  15318112
Total                         315378       15307354             5231  15627963


#### AGE INVALIDE 

In [19]:
panel_solde_df, rep = build_age_valide(
    panel_solde_df,
    strategy="recent",      # ou "mode"
    do_impute_age=True,     # calcule aussi AGE_IMPUTE
    return_tables=True,     # renvoie les tableaux sexe (avant/après)
    debug=False
)

# Afficher les tableaux
print(pd.DataFrame({"Effectif": rep["tab_sexe_avant"], "Pourcentage (%)": rep["tab_sexe_avant_pct"]}))
print(pd.DataFrame({"Effectif": rep["tab_sexe_apres"],  "Pourcentage (%)": rep["tab_sexe_apres_pct"]}))
print(rep["crosstab_sexe"])

# Aperçu des colonnes ajoutées
panel_solde_df[[
    "MATRICULE","DATE_NAISSANCE_CORRIGEE",
    "ANNEE_COLLECTE","MOIS_COLLECTE",
    "SEXE","SEXE_VALIDE",
    "AGE","AGE_VALIDE","AGE_IMPUTE"
]].head()


          Effectif  Pourcentage (%)
SEXE                               
Féminin    4683169            29.97
Masculin  10944644            70.03
None           150             0.00
             Effectif  Pourcentage (%)
SEXE_VALIDE                           
Féminin       4683169            29.97
Masculin     10944794            70.03
SEXE_VALIDE  Féminin  Masculin       Total
SEXE                                      
Féminin      4683169         0   4683169.0
Masculin           0  10944644  10944644.0
NaN                0       150         NaN
Total        4683169  10944794  15627963.0


,MATRICULE,DATE_NAISSANCE_CORRIGEE,ANNEE_COLLECTE,MOIS_COLLECTE,SEXE,SEXE_VALIDE,AGE,AGE_VALIDE,AGE_IMPUTE
0,011242X,1939-01-01,2015,1,Masculin,Masculin,76.0,<NA>,41.0
1,012856Q,1939-01-01,2015,1,Masculin,Masculin,76.0,<NA>,41.0
2,013454N,1924-01-01,2015,1,Masculin,Masculin,91.0,<NA>,41.0
3,027861L,1943-03-01,2015,1,Masculin,Masculin,71.0,<NA>,41.0
4,039659M,1945-12-01,2015,1,Masculin,Masculin,69.0,69.0,69.0


#### AGE DE PRISE DE SERVICE 

In [21]:
panel_solde_df, rep_srv = build_age_service(
    panel_solde_df,
    matricule_col="MATRICULE",
    start_col_raw="PRISE DE SERVICE",
    start_col_corrected="PRISE_DE_SERVICE_CORRIGEE",
    birth_col="DATE_NAISSANCE_CORRIGEE",
    collect_col="DATE_COLLECTE",
    sex_valid_col="SEXE_VALIDE",
    recompute_corrected=True,   # recalcule la date corrigée depuis la collecte la + récente
    age_min=18, age_max=40,
    days_per_year=365,
    return_tables=True
)

# Aperçu des colonnes ajoutées
panel_solde_df[[
    "MATRICULE","DATE_NAISSANCE_CORRIGEE",
   "AGE_AU_SERVICE_IMPUTE","AGE_IMPUTE"
]].head()


,MATRICULE,DATE_NAISSANCE_CORRIGEE,AGE_AU_SERVICE_IMPUTE,AGE_IMPUTE
0,011242X,1939-01-01,30.0,41.0
1,012856Q,1939-01-01,30.0,41.0
2,013454N,1924-01-01,27.0,41.0
3,027861L,1943-03-01,27.0,41.0
4,039659M,1945-12-01,27.0,69.0


#### ANCIENNETE

In [27]:
# 1) Exécuter la fonction et récupérer le DataFrame enrichi
panel_solde_df, rep_anc = build_anciennete(
    panel_solde_df,
    org_col="ID_ORGANISME",          # ou "CODE_ORGANISME"
    start_service_col="PRISE_DE_SERVICE_CORRIGEE",
    adjust_entry_with_service=True,  # cohérence: entrée org >= prise de service
    clip_to_function=True,           # sécurité
    broadcast_latest_org=True,       # crée ANCIENNETE_ORGANISME_MAJ (dernière valeur du couple)
    keep_raw=True,
    return_tables=True
)

# 2) Aperçu (la colonne ANCIENNETE_ORGANISME_MAJ existe désormais même si certains groupes n'ont pas de collecte)
print("Lignes avec DATE_COLLECTE manquante :", rep_anc["nb_lignes_collecte_manquante"])
print(rep_anc["tab_anc_org"].head())

panel_solde_df[[
    "MATRICULE","ID_ORGANISME","DATE_COLLECTE",
    "DATE_ENTREE_ORGANISME","ANCIENNETE_ORGANISME","ANCIENNETE_ORGANISME_MAJ",
    "AGE_IMPUTE","AGE_AU_SERVICE_IMPUTE","ANCIENNETE_DEPUIS_FONCTION"
]].head()


Lignes avec DATE_COLLECTE manquante : 0
ANCIENNETE_ORGANISME
0.0    3442839
1.0    3046843
2.0    2719925
3.0    2384605
4.0    2145250
Name: count, dtype: Int64


,MATRICULE,ID_ORGANISME,DATE_COLLECTE,DATE_ENTREE_ORGANISME,ANCIENNETE_ORGANISME,ANCIENNETE_ORGANISME_MAJ,AGE_IMPUTE,AGE_AU_SERVICE_IMPUTE,ANCIENNETE_DEPUIS_FONCTION
0,011242X,15,2015-01-01,2015-01-01,0.0,4.0,41.0,30.0,11.0
1,012856Q,15,2015-01-01,2015-01-01,0.0,4.0,41.0,30.0,11.0
2,013454N,15,2015-01-01,2015-01-01,0.0,4.0,41.0,27.0,14.0
3,027861L,25,2015-01-01,2015-01-01,0.0,2.0,41.0,27.0,14.0
4,039659M,38,2015-01-01,2015-01-01,0.0,4.0,69.0,27.0,42.0
